# Ch 16　Agentic RAG：用 LangGraph 打造檢索型 agent

> **本 notebook 用離線假零件示範 agentic RAG 的「結構與決策路徑」，免 API key。** 把假檢索器、假分類換成真的 embeddings + LLM，即是線上版（見 .md）。

In [ ]:
!uv pip install -q langgraph

## agentic RAG 的決策骨架：要不要檢索 → 檢索夠不夠 → 不夠就重寫再查

In [ ]:
from typing import Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

# 假知識庫（真實情境是 vector store）
KB = {"reward hacking": "Reward hacking 指 agent 鑽獎勵函數的漏洞來拿高分。",
      "hallucination": "幻覺指模型自信地編造不實內容。"}

class RAGState(TypedDict):
    question: str
    docs: str
    answer: str
    rewrites: int

def decide(state) -> Literal["retrieve", "respond"]:
    # 假 routing：打招呼就直接回，其餘去檢索（線上版是模型用 bind_tools 自己決定）
    return "respond" if "你好" in state["question"] else "retrieve"

def retrieve(state):
    hit = next((v for k, v in KB.items() if k in state["question"]), "")  # 假檢索：關鍵字比對
    return {"docs": hit}

def grade_or_rewrite(state) -> Literal["generate", "rewrite"]:
    # 假評分：檢索到東西就生成；沒有就重寫（但設上限，避免無限改寫）
    if state["docs"]:
        return "generate"
    return "rewrite" if state["rewrites"] < 2 else "generate"

def rewrite(state):
    # 假改寫：補上關鍵字讓下一次檢索更可能命中（線上版是請模型改寫問題）
    return {"question": state["question"] + " reward hacking", "rewrites": state["rewrites"] + 1}

def generate(state):
    src = state["docs"] or "（查無資料）"
    return {"answer": f"根據檢索結果生成答案：{src}"}

def respond(state):
    return {"answer": "你好！需要我幫你查什麼嗎？（不需檢索，直接回答）"}

g = (StateGraph(RAGState)
     .add_node("retrieve", retrieve).add_node("rewrite", rewrite)
     .add_node("generate", generate).add_node("respond", respond)
     .add_conditional_edges(START, decide, {"retrieve": "retrieve", "respond": "respond"})
     .add_conditional_edges("retrieve", grade_or_rewrite, {"generate": "generate", "rewrite": "rewrite"})
     .add_edge("rewrite", "retrieve")      # ← evaluator-optimizer 迴圈：改寫後再檢索
     .add_edge("generate", END).add_edge("respond", END)
     .compile())

def ask(q):
    out = g.invoke({"question": q, "docs": "", "answer": "", "rewrites": 0})
    print(f"Q: {q}\n   → {out['answer']}（改寫 {out['rewrites']} 次）\n")

ask("reward hacking 有哪些類型？")   # 直接命中 → 生成
ask("你好")                          # 打招呼 → 直接回，不檢索
ask("獎勵漏洞是什麼？")              # 首次沒命中 → 改寫補關鍵字 → 再檢索命中 → 生成

## 重點回顧

agentic RAG = RAG + routing（要不要檢索）+ evaluator-optimizer（評分、不夠就重寫再查）。把上面的假零件換成 `bind_tools` 的模型、真 embeddings、structured-output 評分，就是 .md 裡的線上版。